In [24]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from PIL import Image
import sys

### Структурний елемент

In [25]:
INPUT_IMAGE_PATH = "paint_flower2.png"
RESULTS_DIR = "results_morphology"

STRUCTURAL_ELEMENT = np.array([
    [1, 1, 1],
    [1, 1, 1],
    [1, 1, 1]
], dtype=np.uint8)

N_ITERATIONS = 5 # Кількість ітерацій для збільшення сили впливу

#### Допоміжні функції

In [26]:
def ensure_dir(directory):
    if not os.path.exists(directory):
        os.makedirs(directory)

def load_binary_image(path):
    try:
        img = Image.open(path).convert('L') 
        img_array = np.array(img, dtype=np.uint8)
        binary_array = (img_array > 127).astype(np.uint8) 
        return binary_array * 255
    except FileNotFoundError:
        print(f"Помилка: Файл {path} не знайдено.")
        sys.exit(1)
    except Exception as e:
        print(f"Помилка завантаження зображення: {e}")
        sys.exit(1)

def save_image(array_255, filename):
    # ... (Ваша функція save_image)
    img = Image.fromarray(array_255.astype(np.uint8))
    img.save(os.path.join(RESULTS_DIR, filename))
    print(f"✅ Результат '{filename}' збережено у папці '{RESULTS_DIR}'.")

#### Основні функції

##### Розширення

In [27]:
def dilation_single(image, se):
    H, W = image.shape
    h, w = se.shape
    pad_y, pad_x = h // 2, w // 2
    padded_image = np.pad(image, ((pad_y, pad_y), (pad_x, pad_x)), mode='constant', constant_values=0)

    output = np.zeros_like(image, dtype=np.uint8)
    image_bin = (padded_image == 255)
    se_bin = (se == 1)
    
    for i in range(H):
        for j in range(W):
            y_start = i 
            y_end = i + h
            x_start = j 
            x_end = j + w
            
            local_region = image_bin[y_start:y_end, x_start:x_end]
            
            if np.all(local_region[se_bin]):
                output[i, j] = 255
    return output



##### Ерозія

In [28]:
def erosion_single(image, se):
    H, W = image.shape
    h, w = se.shape
    pad_y, pad_x = h // 2, w // 2
    padded_image = np.pad(image, ((pad_y, pad_y), (pad_x, pad_x)), mode='constant', constant_values=0)
    output = np.zeros_like(image, dtype=np.uint8)
    image_bin = (padded_image == 255)
    se_bin = (se == 1)
    
    for i in range(H):
        for j in range(W):
            y_start = i 
            y_end = i + h
            x_start = j 
            x_end = j + w
            
            local_region = image_bin[y_start:y_end, x_start:x_end]
            intersection = local_region & se_bin
            
            if np.any(intersection):
                output[i, j] = 255
    return output

#### Ітерактивне повторення функцій 

In [29]:
def erosion_iterative(image, se, iterations):
    """Ерозія, повторена N разів."""
    current_image = image.copy()
    for _ in range(iterations):
        current_image = erosion_single(current_image, se)
    return current_image

def dilation_iterative(image, se, iterations):
    """Нарощування, повторене N разів."""
    current_image = image.copy()
    for _ in range(iterations):
        current_image = dilation_single(current_image, se)
    return current_image

#### Opening

In [30]:
def opening(image, se, iterations):
    """Розмикання: N ітерацій Ерозії, потім N ітерацій Нарощування."""
    eroded = erosion_iterative(image, se, iterations*2)
    opened = dilation_iterative(eroded, se, iterations*2)
    return opened

##### Closing

In [31]:
def closing(image, se, iterations):
    """Замикання: N ітерацій Нарощування, потім N ітерацій Ерозії."""
    dilated = dilation_iterative(image, se, iterations*2)
    closed = erosion_iterative(dilated, se, iterations*2)
    return closed

##### Boundaries

In [32]:
def boundaries(image, se):
    eroded_once = dilation_single(image, se)
    image_bin = (image == 255)
    eroded_bin = (eroded_once == 255)
    boundary_bin = image_bin & (~eroded_bin)
    return boundary_bin.astype(np.uint8) * 255

#### Main

In [33]:
ensure_dir(RESULTS_DIR)
    
# 0. Завантаження та підготовка
print(f"--- 🚀 Початок лабораторної роботи ({N_ITERATIONS}x сила) ---")
print(f"Застосовуємо морфологічні фільтри ітеративно: {N_ITERATIONS} разів.")
input_image_255 = load_binary_image(INPUT_IMAGE_PATH)
print(f"Розмір зображення: {input_image_255.shape}")
save_image(input_image_255, "0_original.png")

# --- 1. Erosion (Ерозія) ---
print("\n--- 1. Виконання  Ерозії ---")
erosion_result = erosion_iterative(input_image_255, STRUCTURAL_ELEMENT, N_ITERATIONS)
save_image(erosion_result, "1_erosion.png")
    
# --- 2. Dilation (Нарощування) ---
print("\n--- 2. Виконання  Нарощування ---")
dilation_result = dilation_iterative(input_image_255, STRUCTURAL_ELEMENT, N_ITERATIONS)
save_image(dilation_result, "2_dilation.png")

# --- 3. Opening (Розмикання) ---
print("\n--- 3. Виконання  Розмикання (Erosion + Dilation) ---")
opening_result = opening(input_image_255, STRUCTURAL_ELEMENT, N_ITERATIONS)
save_image(opening_result, "3_opening.png")
    
# --- 4. Closing (Замикання) ---
print("\n--- 4. Виконання  Замикання (Dilation + Erosion) ---")
closing_result = closing(input_image_255, STRUCTURAL_ELEMENT, N_ITERATIONS)
save_image(closing_result, "4_closing.png")
    
# --- 5. Границі ---
print("\n--- 5. Виділення Границь (1x Ерозія) ---")
boundaries_result = boundaries(input_image_255, STRUCTURAL_ELEMENT) 
save_image(boundaries_result, "5_boundaries.png")
    
print("\n--- ✅ Лабораторну роботу завершено ---")
print(f"Усі результати збережено у папці '{RESULTS_DIR}'.")
    

--- 🚀 Початок лабораторної роботи (5x сила) ---
Застосовуємо морфологічні фільтри ітеративно: 5 разів.
Розмір зображення: (1094, 1641)
✅ Результат '0_original.png' збережено у папці 'results_morphology'.

--- 1. Виконання  Ерозії ---
✅ Результат '1_erosion.png' збережено у папці 'results_morphology'.

--- 2. Виконання  Нарощування ---
✅ Результат '2_dilation.png' збережено у папці 'results_morphology'.

--- 3. Виконання  Розмикання (Erosion + Dilation) ---
✅ Результат '3_opening.png' збережено у папці 'results_morphology'.

--- 4. Виконання  Замикання (Dilation + Erosion) ---
✅ Результат '4_closing.png' збережено у папці 'results_morphology'.

--- 5. Виділення Границь (1x Ерозія) ---
✅ Результат '5_boundaries.png' збережено у папці 'results_morphology'.

--- ✅ Лабораторну роботу завершено ---
Усі результати збережено у папці 'results_morphology'.
